# DEG analysis: WT pH 5.5 (HCl) vs WT pH 7

Project: **yTF 3** (PRECISE-1K)

| group | condition | sample IDs |
|---|---|---|
| control | WT pH7 | p1k_00809, p1k_00810 |
| treatment | WT pH 5.5 (HCl acidified) | p1k_00815, p1k_00816 |

Shared context: strain MG1655, 37 °C, M9 / glucose. The differing factor is acidification with HCl.

Note: replicates are n=2 per group, so this is a screening (condition-specific expression shift), not a strict differential expression call.

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind
import matplotlib.pyplot as plt

EXPR_PATH = "data/e_coli_precise1k_expression.csv"
META_PATH = "metadata/e_coli_precise1k_samples.csv"
RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

control_samples   = ["p1k_00809", "p1k_00810"]
treatment_samples = ["p1k_00815", "p1k_00816"]
comparison_name   = "WT_pH55_HCl_vs_WT_pH7"

## 1. Sanity-check the chosen samples against the metadata

Confirm strain, temperature, carbon source, media, and pH match the intended comparison.

In [ ]:
meta = pd.read_csv(META_PATH)
chosen = control_samples + treatment_samples
cols = ["unique_identifier", "project_name", "condition", "pH",
        "reference_strain", "Temperature (C)", "Carbon Source (g/L)",
        "Base Media", "n_replicates", "reference_condition_identifier"]
meta_sub = meta[meta["unique_identifier"].isin(chosen)][cols].set_index("unique_identifier").loc[chosen]
meta_sub

## 2. Load expression matrix and pull out the four columns

PRECISE-1K's expression matrix is delivered as **log-TPM (log2 scale, mean-centered per gene)**: values span roughly -12 to +13 with negatives present. So `treatment_mean - control_mean` is already a log2 fold change.

In [ ]:
expr = pd.read_csv(EXPR_PATH, index_col=0)
print("expression matrix shape:", expr.shape)
print("value range:", float(expr.values.min()), "→", float(expr.values.max()))

missing = [s for s in chosen if s not in expr.columns]
assert not missing, f"missing samples in expression matrix: {missing}"

control   = expr[control_samples]
treatment = expr[treatment_samples]
control.head()

## 3. Compute log2FC, Welch t-test p-value, regulation call

In [ ]:
control_mean   = control.mean(axis=1)
treatment_mean = treatment.mean(axis=1)
log2fc = treatment_mean - control_mean

# Welch t-test per gene (n=2 vs n=2 -> very low power, used for screening only)
tvals, pvals = ttest_ind(treatment.values, control.values, axis=1, equal_var=False)

deg = pd.DataFrame({
    "gene": expr.index,
    "control_mean":   control_mean.values,
    "treatment_mean": treatment_mean.values,
    "log2FC":  log2fc.values,
    "p_value": pvals,
})
deg["p_value"]     = deg["p_value"].fillna(1.0).replace(0, np.nextafter(0, 1))
deg["neg_log10_p"] = -np.log10(deg["p_value"])

deg["regulation"] = "ns"
deg.loc[(deg["log2FC"] >=  1) & (deg["p_value"] < 0.05), "regulation"] = "up"
deg.loc[(deg["log2FC"] <= -1) & (deg["p_value"] < 0.05), "regulation"] = "down"

deg = deg.sort_values("p_value").reset_index(drop=True)

deg_path = os.path.join(RESULTS_DIR, f"DEG_{comparison_name}.csv")
deg.to_csv(deg_path, index=False)

print("genes:", len(deg))
print(deg["regulation"].value_counts().to_dict())
print("saved:", deg_path)
deg.head(10)

## 4. Volcano plot

In [ ]:
color_map = {"up": "#d62728", "down": "#1f77b4", "ns": "#bdbdbd"}
colors = deg["regulation"].map(color_map)

fig, ax = plt.subplots(figsize=(9, 7), dpi=150)
ax.scatter(deg["log2FC"], deg["neg_log10_p"], s=14, alpha=0.75, c=colors, edgecolors="none")
# Add a little horizontal headroom so labels stay inside the plot area
xmin, xmax = deg["log2FC"].min(), deg["log2FC"].max()
ax.set_xlim(xmin - 0.6, xmax + 0.6)

ax.axvline( 1, linestyle="--", linewidth=1, color="black")
ax.axvline(-1, linestyle="--", linewidth=1, color="black")
ax.axhline(-np.log10(0.05), linestyle="--", linewidth=1, color="black")

# Label top 5 up / top 5 down significant DEGs.
# Down hits cluster tightly around log2FC ~ -1.6, so we stagger their text
# offsets to keep all 10 labels (incl. ldtC) readable.
gene_info = pd.read_csv("metadata/gene_info.csv", usecols=["locus_tag", "gene_name", "gene_product"])
b2name = dict(zip(gene_info["locus_tag"], gene_info["gene_name"]))

top_up   = deg[deg["regulation"] == "up"  ].nlargest(5, "log2FC").copy()
top_down = deg[deg["regulation"] == "down"].nsmallest(5, "log2FC").copy()

label_kwargs = dict(fontsize=9, fontweight="bold", textcoords="offset points")

for _, r in top_up.iterrows():
    label = b2name.get(r["gene"], r["gene"]) or r["gene"]
    ax.annotate(label, (r["log2FC"], r["neg_log10_p"]),
                xytext=(6, 3), ha="left", color="#7a0a0a", **label_kwargs)

# Manual per-gene offsets to keep the tightly clustered down hits readable.
# All offsets push labels to the RIGHT (positive dx) so they stay inside the
# plot area and never overlap with the y-axis tick labels.
down_offsets = {
    "malM": ( 14,  10),
    "malK": ( 40, -10),
    "lamB": ( 70,   0),
    "malE": ( 60, -22),
    "ldtC": ( 18,  18),
}
for _, r in top_down.iterrows():
    label = b2name.get(r["gene"], r["gene"]) or r["gene"]
    dx, dy = down_offsets.get(label, (10, 6))
    ax.annotate(label, (r["log2FC"], r["neg_log10_p"]),
                xytext=(dx, dy), ha="left", color="#0a2a5e",
                arrowprops=dict(arrowstyle="-", color="#0a2a5e",
                                lw=0.6, alpha=0.7),
                **label_kwargs)

ax.set_xlabel("log2 fold change (pH 5.5 HCl / pH 7)")
ax.set_ylabel("-log10(p-value)")
ax.set_title("Acidic pH stress induces a strong transcriptional response\nin E. coli MG1655 (pH 5.5 HCl vs pH 7)")

volcano_path = os.path.join(RESULTS_DIR, f"volcano_{comparison_name}.png")
plt.tight_layout()
plt.savefig(volcano_path, dpi=600, bbox_inches="tight")
plt.show()
print("saved:", volcano_path)

## 5. Top 5 significant up / down with gene-name annotation

For "top 5 down" we restrict to genes that pass the DEG cutoff (|log2FC| ≥ 1 AND p < 0.05). This drops borderline hits like b1599 (mdtI, p = 0.056) from the down list. Gene name and gene product are joined from the PRECISE-1K `gene_info.csv` (NCBI RefSeq NC_000913.3).

In [ ]:
gene_info = pd.read_csv("metadata/gene_info.csv",
                        usecols=["locus_tag", "gene_name", "gene_product"])

top5_up = (
    deg[deg["regulation"] == "up"]
    .sort_values("log2FC", ascending=False)
    .head(5)
    .assign(direction="up")
)
top5_down = (
    deg[deg["regulation"] == "down"]
    .sort_values("log2FC", ascending=True)
    .head(5)
    .assign(direction="down")
)
top5 = pd.concat([top5_up, top5_down], ignore_index=True)

top5 = top5.merge(gene_info, left_on="gene", right_on="locus_tag", how="left")
top5 = top5[["direction", "gene", "gene_name", "gene_product",
             "log2FC", "p_value", "neg_log10_p",
             "control_mean", "treatment_mean", "regulation"]]
top5 = top5.rename(columns={"gene": "b_number"})

top5_path = os.path.join(RESULTS_DIR, f"top5_significant_up_down_{comparison_name}.csv")
top5.to_csv(top5_path, index=False)
print("saved:", top5_path)
top5

## 6. Slide-ready table figures

Render the comparison-setup table and the top-5 up/down table as standalone PNGs that can be dropped directly into a PowerPoint slide.

In [ ]:
def render_table_png(df, path, title=None, col_widths=None,
                     row_colors=None, header_color="#2c3e50",
                     header_text_color="white",
                     fig_width=14, row_height=0.55,
                     fontsize=12):
    """Render a pandas DataFrame as a PNG table image (slide-ready).

    `col_widths` should sum to ~1.0 (fractions of the axes width).
    `fig_width` controls how wide the figure is in inches; widen this
    for tables with long text so cells don't visually overflow.
    """
    n_rows, n_cols = df.shape
    fig_height = row_height * (n_rows + 1) + (0.7 if title else 0.2)

    fig, ax = plt.subplots(figsize=(fig_width, fig_height), dpi=600)
    ax.axis("off")

    if title:
        ax.set_title(title, fontsize=fontsize + 2, fontweight="bold",
                     loc="left", pad=8)

    tbl = ax.table(
        cellText=df.astype(str).values.tolist(),
        colLabels=df.columns.tolist(),
        colWidths=col_widths,
        cellLoc="center",
        loc="center",
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(fontsize)
    tbl.scale(1, 1.5)

    # Header styling
    for j in range(n_cols):
        c = tbl[(0, j)]
        c.set_facecolor(header_color)
        c.set_text_props(color=header_text_color, fontweight="bold")
        c.set_edgecolor("white")
        c.set_height(c.get_height() * 1.05)

    # Body row styling
    for i in range(n_rows):
        bg = (row_colors[i] if row_colors is not None
              else ("#f5f7fa" if i % 2 == 0 else "white"))
        for j in range(n_cols):
            c = tbl[(i + 1, j)]
            c.set_facecolor(bg)
            c.set_edgecolor("white")

    plt.savefig(path, dpi=600, bbox_inches="tight",
                facecolor="white", pad_inches=0.15)
    plt.show()
    print("saved:", path)

In [ ]:
setup_df = pd.DataFrame([
    {"group": "control",   "condition": "WT pH 7",         "sample IDs": "p1k_00809, p1k_00810",
     "strain": "MG1655",   "temperature": "37 °C", "media / carbon": "M9 / glucose"},
    {"group": "treatment", "condition": "WT pH 5.5 (HCl)", "sample IDs": "p1k_00815, p1k_00816",
     "strain": "MG1655",   "temperature": "37 °C", "media / carbon": "M9 / glucose"},
])

setup_row_colors = ["#eaf2fb", "#fdecec"]
setup_path = os.path.join(RESULTS_DIR, "comparison_setup_WT_pH55_HCl_vs_WT_pH7.png")

render_table_png(
    setup_df,
    setup_path,
    title="Comparison setup  (yTF 3 project, PRECISE-1K)",
    col_widths=[0.10, 0.18, 0.27, 0.10, 0.14, 0.18],
    row_colors=setup_row_colors,
    fig_width=15,
    fontsize=13,
)

In [ ]:
from urllib.parse import unquote

top5_fig = top5.copy()
# Decode URL-encoded chars (e.g. "%2C" -> ",") that come from the source CSV
top5_fig["gene_product"] = top5_fig["gene_product"].fillna("").map(unquote)
# Trim "...protein YfdX" style redundancy and shorten long products for slide use
def shorten(s, limit=68):
    s = s.replace("DNA-binding transcriptional dual regulator",
                  "transcriptional dual regulator")
    s = s.replace("uncharacterized protein ", "uncharacterized: ")
    return s if len(s) <= limit else s[: limit - 1] + "…"
top5_fig["gene_product"] = top5_fig["gene_product"].map(shorten)

top5_fig["log2FC"]  = top5_fig["log2FC"].map(lambda v: f"{v:+.2f}")
top5_fig["p_value"] = top5_fig["p_value"].map(lambda v: f"{v:.2e}")
top5_fig = top5_fig[["direction", "b_number", "gene_name",
                     "gene_product", "log2FC", "p_value"]]
top5_fig.columns = ["direction", "b-number", "gene name",
                    "gene product", "log2FC", "p-value"]

row_colors = ["#fdecec" if d == "up" else "#eaf2fb"
              for d in top5_fig["direction"]]
top5_path = os.path.join(RESULTS_DIR, "top5_table_WT_pH55_HCl_vs_WT_pH7.png")

render_table_png(
    top5_fig,
    top5_path,
    title="Top 5 significant up- and down-regulated DEGs  "
          "(|log2FC| ≥ 1, p < 0.05)",
    col_widths=[0.08, 0.09, 0.09, 0.55, 0.09, 0.10],
    row_colors=row_colors,
    fig_width=20,
    fontsize=12,
)

## Interpretation notes

This comparison captures the transcriptional response of WT *E. coli* MG1655 to acidic pH stress, using the matched WT pH 7 condition as the reference. The DEG result shows a strong induction-biased response, with 78 up-regulated genes and 19 down-regulated genes under the threshold of |log2FC| ≥ 1 and p-value < 0.05.

The up-regulated genes are broadly consistent with acid stress biology. In particular, **asr** encodes acid shock protein, and **ydeO** is associated with acid resistance regulation. These genes support the interpretation that the pH 5.5 HCl condition activates an acid stress response.

However, some gene-level interpretations should be kept conservative. **yegR** is annotated as an uncharacterized protein and should not be over-interpreted. **frc** is annotated as formyl-CoA transferase and is more safely described as related to oxalate/formate-associated metabolism rather than as a direct acid resistance gene. **yfdX** should be described as a gene located in an acid-resistance-associated genomic neighborhood, rather than as a fully established acid resistance factor.

The down-regulated genes show a coherent decrease in the maltose transport regulon, including **malK**, **lamB**, **malE**, and **malM**. This suggests that acidic pH stress is accompanied by repression of maltose uptake-related functions under the tested M9-glucose condition.

Because the comparison uses n=2 replicates per condition, the result should be interpreted as a condition-specific expression shift screening rather than a definitive high-powered DEG analysis.

---

### Interpretation guardrails (slide-ready short form)

- **asr** and **ydeO** provide clear support for acid stress response activation.
- **yegR** is uncharacterized; no strong functional claim was made.
- **frc** was interpreted conservatively as oxalate/formate-associated metabolism.
- **yfdX** was described as acid-resistance-neighborhood-associated, not as a confirmed acid resistance gene.
- Because each condition has n=2 replicates, this analysis is best interpreted as condition-specific expression shift screening.